# TD — SOLID / SRP(Single Responsability Principle)

---

## Objectifs

À la fin de ce TD, vous serez capables de :

- expliquer le principe **SRP** ;
- identifier une classe ou une fonction qui a **trop de responsabilités** ;
- refactorer un script de scraping pour séparer les responsabilités ;
- construire un pipeline simple de scraping plus **lisible**, **testable** et **maintenable**.

---

## Rappel — Qu'est-ce que le SRP ?

Le **Single Responsibility Principle** signifie :

> **Une classe, une fonction ou un module ne doit avoir qu'une seule raison de changer.**

En contexte de scraping, c'est très fréquent de voir un seul script qui :
- télécharge une page,
- parse le HTML,
- nettoie les données,
- gère les erreurs,
- exporte en CSV,
- affiche des logs.

Le SRP consiste à **séparer ces responsabilités**.

---

## Contexte du TD

Nous allons travailler sur un petit cas de scraping à partir d'un HTML simulé représentant une liste de livres.
Le but n'est pas de dépendre d'un vrai site externe, mais de **se concentrer sur la conception logicielle**.

Ensuite, vous pourrez facilement adapter cette architecture à un vrai site.

In [1]:
# Dépendances du TD
# BeautifulSoup sera utilisée pour parser le HTML.
# Si nécessaire, décommentez la ligne suivante :
# !pip install beautifulsoup4 pandas requests lxml

from bs4 import BeautifulSoup
import pandas as pd
from pathlib import Path
import textwrap

## 1. Jeu de données HTML simulé

Dans un vrai projet, ce HTML proviendrait d'une requête HTTP.
Ici, on le stocke localement pour travailler sans dépendre d'Internet.

In [2]:
HTML_SAMPLE = '''
<html>
  <body>
    <section class="catalog">
      <article class="book">
        <h2 class="title">Clean Code</h2>
        <p class="price">32.50 €</p>
        <p class="rating">4.7</p>
      </article>
      <article class="book">
        <h2 class="title">Designing Data-Intensive Applications</h2>
        <p class="price">45.00 €</p>
        <p class="rating">4.9</p>
      </article>
      <article class="book">
        <h2 class="title">Python for Data Analysis</h2>
        <p class="price">38.90 €</p>
        <p class="rating">4.5</p>
      </article>
    </section>
  </body>
</html>
'''
print(HTML_SAMPLE[:300])


<html>
  <body>
    <section class="catalog">
      <article class="book">
        <h2 class="title">Clean Code</h2>
        <p class="price">32.50 €</p>
        <p class="rating">4.7</p>
      </article>
      <article class="book">
        <h2 class="title">Designing Data-Intensive Applications</


## 2. Première version : un script qui viole le SRP

La fonction suivante fait **trop de choses à la fois** :

1. récupère les données source ;
2. parse le HTML ;
3. extrait les livres ;
4. transforme les prix ;
5. construit un DataFrame ;
6. sauvegarde en CSV ;
7. affiche un résumé.

### Travail demandé

1. Lisez le code.
2. Identifiez les différentes responsabilités mélangées.
3. Discutez : quelles sont les conséquences si une seule étape change ?

In [3]:
def scrape_books_bad(html: str, output_file: str = "books_bad.csv") -> pd.DataFrame:
    # 1) Parsing HTML
    soup = BeautifulSoup(html, "html.parser")

    # 2) Extraction
    books = []
    for article in soup.select("article.book"):
        title = article.select_one(".title").get_text(strip=True)
        price_raw = article.select_one(".price").get_text(strip=True)
        rating_raw = article.select_one(".rating").get_text(strip=True)

        # 3) Nettoyage / transformation
        price = float(price_raw.replace("€", "").replace(",", ".").strip())
        rating = float(rating_raw)

        books.append({
            "title": title,
            "price_eur": price,
            "rating": rating
        })

    # 4) Construction DataFrame
    df = pd.DataFrame(books)

    # 5) Export
    df.to_csv(output_file, index=False)

    # 6) Reporting
    print(f"{len(df)} livres exportés vers {output_file}")
    print(f"Prix moyen : {df['price_eur'].mean():.2f} €")

    return df

df_bad = scrape_books_bad(HTML_SAMPLE)
df_bad

3 livres exportés vers books_bad.csv
Prix moyen : 38.80 €


,title,price_eur,rating
0,Clean Code,32.5,4.7
1,Designing Data-Intensive Applications,45.0,4.9
2,Python for Data Analysis,38.9,4.5


## 3. Analyse de la violation du SRP

### Question 1
Pourquoi cette fonction viole-t-elle le SRP ? </br>
La fonction scrape_books_bad viole le SRP car elle a plusieurs responsabilités : parsing du HTML, extraction des données, nettoyage des données, création du DataFrame, export en CSV, affichage des résultats. Donc elle a plusieurs raisons de changer, ce qui va à l’encontre du principe SRP.

### Indice
Demandez-vous :
- si la source change (fichier, API, page web), faut-il modifier cette fonction ?
- si le format HTML change, faut-il modifier cette fonction ?
- si le format de sortie change (CSV, JSON, base SQL), faut-il modifier cette fonction ?
- si la logique de nettoyage change, faut-il modifier cette fonction ?

## 4. Refactoring guidé — séparation des responsabilités

Nous allons maintenant séparer le code en plusieurs composants :

- **HtmlProvider** : fournit le HTML ;
- **BookParser** : extrait les données brutes depuis le HTML ;
- **BookCleaner** : transforme et nettoie les champs ;
- **BookRepository** : sauvegarde les résultats ;
- **BookReporter** : affiche un résumé ;
- **ScrapingPipeline** : orchestre l'ensemble.

Chaque composant a **une responsabilité principale**.

In [4]:
class HtmlProvider:
    """Responsable de fournir le HTML brut."""

    def get_html(self) -> str:
        return HTML_SAMPLE


class BookParser:
    """Responsable d'extraire les données textuelles depuis le HTML."""

    def parse(self, html: str) -> list[dict]:
        soup = BeautifulSoup(html, "html.parser")
        books = []

        for article in soup.select("article.book"):
            books.append({
                "title": article.select_one(".title").get_text(strip=True),
                "price_raw": article.select_one(".price").get_text(strip=True),
                "rating_raw": article.select_one(".rating").get_text(strip=True),
            })

        return books


class BookCleaner:
    """Responsable de nettoyer et convertir les données extraites."""

    def clean(self, raw_books: list[dict]) -> list[dict]:
        cleaned = []

        for book in raw_books:
            cleaned.append({
                "title": book["title"],
                "price_eur": float(book["price_raw"].replace("€", "").replace(",", ".").strip()),
                "rating": float(book["rating_raw"]),
            })

        return cleaned


class BookRepository:
    """Responsable de persister les données."""

    def save_csv(self, books: list[dict], output_file: str) -> pd.DataFrame:
        df = pd.DataFrame(books)
        df.to_csv(output_file, index=False)
        return df


class BookReporter:
    """Responsable d'afficher des indicateurs simples."""

    def report(self, df: pd.DataFrame) -> None:
        print(f"{len(df)} livres exportés")
        print(f"Prix moyen : {df['price_eur'].mean():.2f} €")
        print(f"Note moyenne : {df['rating'].mean():.2f}")


class ScrapingPipeline:
    """Responsable d'orchestrer les composants."""

    def __init__(self, provider, parser, cleaner, repository, reporter):
        self.provider = provider
        self.parser = parser
        self.cleaner = cleaner
        self.repository = repository
        self.reporter = reporter

    def run(self, output_file: str = "books_clean.csv") -> pd.DataFrame:
        html = self.provider.get_html()
        raw_books = self.parser.parse(html)
        cleaned_books = self.cleaner.clean(raw_books)
        df = self.repository.save_csv(cleaned_books, output_file)
        self.reporter.report(df)
        return df

In [5]:
pipeline = ScrapingPipeline(
    provider=HtmlProvider(),
    parser=BookParser(),
    cleaner=BookCleaner(),
    repository=BookRepository(),
    reporter=BookReporter()
)

df = pipeline.run()
df

3 livres exportés
Prix moyen : 38.80 €
Note moyenne : 4.70


,title,price_eur,rating
0,Clean Code,32.5,4.7
1,Designing Data-Intensive Applications,45.0,4.9
2,Python for Data Analysis,38.9,4.5


## 5. Ce que le refactoring apporte

### Lisibilité
Le code est découpé en blocs compréhensibles.

### Testabilité
On peut tester chaque composant séparément :
- tester le parser sans écrire de CSV ;
- tester le cleaner sans parser du HTML ;
- tester le repository avec des données simulées.

### Maintenabilité
Si le format HTML change, on modifie surtout **BookParser**.  
Si l'export change, on modifie surtout **BookRepository**.

### Réutilisabilité
Le cleaner ou le reporter peuvent être réutilisés dans d'autres pipelines.

## 6. Exercice 1 — compléter un parser spécialisé

On ajoute une catégorie à chaque livre.

### Travail demandé
1. Modifiez le HTML ci-dessous.
2. Complétez la méthode `parse`.
3. Vérifiez que la colonne `category` apparaît dans le DataFrame final.

> Objectif pédagogique : constater qu'un changement de structure HTML impacte surtout le parser.

In [9]:
HTML_WITH_CATEGORY = '''
<html>
  <body>
    <section class="catalog">
      <article class="book">
        <h2 class="title">Clean Code</h2>
        <p class="category">Software Engineering</p>
        <p class="price">32.50 €</p>
        <p class="rating">4.7</p>
      </article>
      <article class="book">
        <h2 class="title">Python for Data Analysis</h2>
        <p class="category">Data</p>
        <p class="price">38.90 €</p>
        <p class="rating">4.5</p>
      </article>
    </section>
  </body>
</html>
'''

class CategoryBookParser:
    def parse(self, html: str) -> list[dict]:
        soup = BeautifulSoup(html, "html.parser")
        books = []

        for article in soup.select("article.book"):
             books.append({
                "title": article.select_one(".title").get_text(strip=True),
                "category": article.select_one(".category").get_text(strip=True),
                "price_raw": article.select_one(".price").get_text(strip=True),
                "rating_raw": article.select_one(".rating").get_text(strip=True),
            })

        return books

# À vous de jouer :
parser = CategoryBookParser()
raw_books = parser.parse(HTML_WITH_CATEGORY)
raw_books

[{'title': 'Clean Code',
  'category': 'Software Engineering',
  'price_raw': '32.50 €',
  'rating_raw': '4.7'},
 {'title': 'Python for Data Analysis',
  'category': 'Data',
  'price_raw': '38.90 €',
  'rating_raw': '4.5'}]

## 7. Exercice 2 — changer uniquement la persistance

On souhaite maintenant sauvegarder en **JSON** au lieu de CSV.

### Travail demandé
1. Créez une nouvelle classe `JsonBookRepository`.
2. Elle doit avoir une seule responsabilité : sauvegarder les données au format JSON.
3. Réutilisez le reste du pipeline sans modifier les autres classes.

> Objectif : illustrer le fait que le changement de format de sortie ne doit pas impacter le parser ni le cleaner.

In [10]:
import json


class JsonBookRepository:
    def save_json(self, books: list[dict], output_file: str):
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(books, f, indent=4, ensure_ascii=False)

        print(f"Fichier JSON sauvegardé : {output_file}")

# Exemple d'usage attendu
json_repo = JsonBookRepository()
json_repo.save_json([{"title": "Clean Code", "price_eur": 32.5, "rating": 4.7}], "books.json")

Fichier JSON sauvegardé : books.json


## 8. Exercice 3 — brancher une vraie source web

Dans un vrai cas de scraping, le fournisseur de HTML peut faire une requête HTTP.

### Travail demandé
1. Lisez le code.
2. Expliquez pourquoi cette classe n'a qu'une responsabilité principale.
3. Indiquez quelle autre classe sera modifiée si le HTML de la page change.

In [ ]:
# Exemple non exécuté par défaut
# import requests

class RemoteHtmlProvider:
    """Responsable d'aller chercher le HTML depuis une URL."""

    def __init__(self, url: str):
        self.url = url

    def get_html(self) -> str:
        # response = requests.get(self.url, timeout=10)
        # response.raise_for_status()
        # return response.text
        raise NotImplementedError("À activer seulement dans un vrai contexte de scraping autorisé.")

### Réponses
2. Expliquez pourquoi cette classe n'a qu'une responsabilité principale.</br>
La classe RemoteHtmlProvider a une seule responsabilité car elle est uniquement dédiée à la récupération du HTML depuis une URL..</br>
3. Indiquez quelle autre classe sera modifiée si le HTML de la page change.<br>
La classe BookParser

#### Supposons cette fois, on veux scrapper sur le site web : http://books.toscrape.com/ ayant plusieurs pages.
4. Lisez le code.
5. Expliquez pourquoi cette classe n'a qu'une responsabilité principale.
6. Expliquer pourquoi la classe RemoteHtmlProvider doit etre modifier et proposer un code.
7. Indiquez quelle autre classe sera modifiée.

In [11]:
from urllib.parse import urljoin

class UrlBuilder:
    """Responsable de construire les URLs."""

    BASE_URL = "http://books.toscrape.com/"

    def get_catalogue_page(self, page: int) -> str:
        return urljoin(self.BASE_URL, f"catalogue/page-{page}.html")

In [14]:
# import requests

class RemoteHtmlProvider:
    """Responsable uniquement de récupérer le HTML."""
    def get_html(self, url: str) -> str:
        response = requests.get(url)
        
        if response.status_code != 200:
            raise Exception("Erreur HTTP")
        
        return response.text

### Réponses
4. Lisez le code.
5. Expliquez pourquoi cette classe n'a qu'une responsabilité principale.</br>
car elle est uniquement dédiée à la construction des urls
6. Expliquer pourquoi la classe RemoteHtmlProvider doit etre modifier et proposer un code.</br>
Parce que maintenant on scrape plusieurs pages donc il faut passer une URL dynamique </br>
7. Indiquez quelle autre classe sera modifiée.</br>
BookParser car le HTML du site réel est différent du HTML simulé

## 9. Exercice 4 — tests unitaires ciblés

Le SRP facilite les tests unitaires.  
Ici, nous testons uniquement `BookCleaner`.

### Travail demandé
1. Exécutez le test.
2. Ajoutez un second cas de test avec une autre entrée.
3. Expliquez pourquoi ce test est simple à écrire grâce au SRP.</br>

Grâce au SRP, BookCleaner : ne dépend pas du HTML, ni du CSV, ni d’Internet donc on peut le tester facilement avec des données simples

In [17]:
def test_book_cleaner():
    cleaner = BookCleaner()
    raw_books = [
        {"title": "Test Book", "price_raw": "10.99 €", "rating_raw": "4.2"}
    ]

    result = cleaner.clean(raw_books)

    assert result == [
        {"title": "Test Book", "price_eur": 10.99, "rating": 4.2}
    ]
    
    # Nouveau test
    raw_books_2 = [
    {"title": "Another Book", "price_raw": "20.00 €", "rating_raw": "3.5"}
    ]

    result_2 = cleaner.clean(raw_books_2)

    assert result_2 == [
    {"title": "Another Book", "price_eur": 20.0, "rating": 3.5}
    ]


test_book_cleaner()
print("Test OK")
print("Test 2 OK")

Test OK
Test 2 OK


**Fin du TD**